# 🎱 4D Number Generator — Statistical + Lucky Feel

**Philosophy:** 4D prizes are fixed amounts — no sharing problem. Every number 0000–9999 has equal probability.  
So the goal here is different from TOTO:

1. **Statistical grounding** — analyse digit patterns from 4,479 draws to understand what the data actually says  
2. **Lucky feel numbers** — you input real-world signs you've witnessed (car plates at accidents, temple chits, dream numbers, etc.) and the notebook derives picks from them  
3. **Smart random** — statistically-weighted random generation that avoids known cold streaks  

**4D draws:** Wednesday, Saturday, Sunday — 3 chances per week.

---

## ⚙️ BLOCK 1 — Config, Imports & Your Lucky Event Numbers

### ✏️ Edit the Lucky Events section below before running
This is the most personal block. Enter any 4-digit numbers you've recently encountered that *felt meaningful* —  
car plate of a vehicle in an accident you witnessed, numbers from a temple fortune stick, a dream, a receipt total, anything.

In [ ]:
import os
import sqlite3
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from collections import Counter
from itertools import permutations
from scipy.stats import chisquare
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')
np.random.seed(None)   # True randomness each run

# ── DB CONFIG ────────────────────────────────────────────────────────────────
DB_PATH   = 'data/4d_results.db'
TOP_N     = 4     # picks per category in final output

# ── ✏️  YOUR LUCKY EVENT NUMBERS — EDIT THIS SECTION ─────────────────────────
# Add any 4-digit numbers you've recently seen and felt were meaningful.
# Format: ('NNNN', 'description of where you saw it')
# Leave empty list [] if you have no lucky events this week.

LUCKY_EVENTS = [
    # Examples (replace or delete these with your own):
    # ('3947', 'Car plate of taxi I took on a good day'),
    # ('0812', 'Temple fortune stick number, Kuan Yin temple'),
    # ('7291', 'Accident car plate I witnessed on CTE'),
    # ('4481', 'Receipt total that felt odd'),
    # ('1188', 'Dream number — appeared twice'),
    ('8888', 'Grandfather Mandai Cremantorium Block address'),
    ('8888', 'Ah gong Ah Ma new "car"'),
    ('1111', 'My Birthday'),
     
]

# ── GENERATION WEIGHTS (how many picks per method) ──────────────────────────
N_PURE_RANDOM   = 5    # completely random 4-digit numbers
N_STAT_WEIGHTED = 5    # weighted by digit frequency from historical data
N_LUCKY_DERIVED = 5    # derived from your LUCKY_EVENTS above
N_HOT_NUMBERS   = 5    # numbers with recent winning streaks
N_OVERDUE       = 5    # numbers that haven't appeared in a long time
# ─────────────────────────────────────────────────────────────────────────────

print('✅ Config loaded.')
print(f'   DB path  : {DB_PATH}')
print(f'   Lucky events entered: {len(LUCKY_EVENTS)}')
for num, desc in LUCKY_EVENTS:
    print(f'     → {num}  "{desc}"')
if not LUCKY_EVENTS:
    print('   (none — add some in LUCKY_EVENTS above for personalised picks)')

---
## 🔍 BLOCK 2 — Load Data

In [ ]:
conn = sqlite3.connect(DB_PATH)

# Schema check
tables = conn.execute("SELECT name FROM sqlite_master WHERE type='table'").fetchall()
print(f'Tables found: {[t[0] for t in tables]}')
print(f'File exists : {__import__("os").path.exists(DB_PATH)}')

df = pd.read_sql('SELECT * FROM draws ORDER BY draw_number ASC', conn)
df['draw_date'] = pd.to_datetime(df['draw_date'], format='%a, %d %b %Y', errors='coerce')

PRIZE_COLS       = ['first_prize', 'second_prize', 'third_prize']
STARTER_COLS     = [f'starter_{i}'     for i in range(1, 11)]
CONSOLATION_COLS = [f'consolation_{i}' for i in range(1, 11)]
ALL_PRIZE_COLS   = PRIZE_COLS + STARTER_COLS + CONSOLATION_COLS

# Flatten all historical prize numbers
all_prize_nums = []
for col in ALL_PRIZE_COLS:
    all_prize_nums.extend(df[col].dropna().tolist())

print(f'\n✅ Loaded {len(df):,} draws')
print(f'   Date range : {df["draw_date"].min().date()} → {df["draw_date"].max().date()}')
print(f'   Prize numbers total: {len(all_prize_nums):,}  ({len(ALL_PRIZE_COLS)} per draw)')
display(df[['draw_number','draw_date'] + PRIZE_COLS].tail(3))

---
## 📊 BLOCK 3 — Digit Frequency Analysis
Are any digits statistically hotter or colder than expected at each position?  
This is what feeds the **stat-weighted random generator** in Block 6.

In [ ]:
# Per-position digit frequency
pos_weights = {}   # will store weight arrays for Block 6

fig, axes = plt.subplots(2, 2, figsize=(14, 7))
axes = axes.flatten()
pos_names = ['Thousands', 'Hundreds', 'Tens', 'Units']

for pos in range(4):
    digits = [n[pos] for n in all_prize_nums if len(str(n)) == 4]
    counts = Counter(digits)
    y      = np.array([counts.get(str(d), 0) for d in range(10)])
    expected = y.sum() / 10

    # Normalise to probability weights for random generation
    pos_weights[pos] = y / y.sum()

    chi2, p = chisquare(y)
    colors   = ['#e74c3c' if abs(v-expected)/expected > 0.015
                else '#3498db' for v in y]

    axes[pos].bar(range(10), y, color=colors, edgecolor='white')
    axes[pos].axhline(expected, color='black', linestyle='--', linewidth=1.5)
    axes[pos].set_xticks(range(10))
    axes[pos].set_xlabel('Digit 0–9')
    axes[pos].set_ylabel('Times appeared')
    status = '⚠️ SKEWED' if p < 0.05 else '✅ UNIFORM'
    axes[pos].set_title(f'{pos_names[pos]} position — {status}\nχ²={chi2:.2f}, p={p:.4f}',
                        fontweight='bold')

plt.suptitle('4D Digit Frequency per Position — All Historical Draws', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('4d_v2_block3_digits.png', dpi=150)
plt.show()

print('\n📋 Position weights (higher = that digit appears more in history):')
for pos in range(4):
    w = pos_weights[pos]
    top3 = sorted(range(10), key=lambda d: w[d], reverse=True)[:3]
    print(f'  {pos_names[pos]:12s}: hottest digits = {top3}  '
          f'coldest = {sorted(range(10), key=lambda d: w[d])[:3]}')

---
## 📈 BLOCK 4 — Hot Numbers & Overdue Numbers
**Hot** = appeared multiple times in recent draws (streaks exist even in random data by chance)  
**Overdue** = haven't appeared in a very long time  

> Neither is more likely to win next. But they feed two separate pick categories in Block 6.

In [ ]:
# All-time win counter per number
win_counter   = Counter(all_prize_nums)
first_counter = Counter(df['first_prize'].dropna().tolist())

# Recent window: last 100 draws
recent_df  = df.tail(100)
recent_nums = []
for col in ALL_PRIZE_COLS:
    recent_nums.extend(recent_df[col].dropna().tolist())
recent_counter = Counter(recent_nums)

# Last-seen draw per number
last_seen_draw = {}
for _, row in df.iterrows():
    dn = row['draw_number']
    for col in ALL_PRIZE_COLS:
        val = row[col]
        if pd.notna(val):
            last_seen_draw[val] = dn

latest_draw = df['draw_number'].max()

# Build lookup for all 10,000 numbers
all_possible = [f'{n:04d}' for n in range(10000)]

num_df = pd.DataFrame({
    'number'          : all_possible,
    'all_time_wins'   : [win_counter.get(n, 0) for n in all_possible],
    'first_prize_wins': [first_counter.get(n, 0) for n in all_possible],
    'recent_100_wins' : [recent_counter.get(n, 0) for n in all_possible],
    'draws_since_seen': [latest_draw - last_seen_draw.get(n, 0)
                         for n in all_possible],
})

hot_numbers     = num_df.nlargest(50,  'recent_100_wins')['number'].tolist()
overdue_numbers = num_df.nlargest(50,  'draws_since_seen')['number'].tolist()
never_won       = num_df[num_df['all_time_wins'] == 0]['number'].tolist()

print(f'Numbers that appeared in last 100 draws: {(num_df["recent_100_wins"]>0).sum():,}')
print(f'Numbers that have NEVER appeared       : {len(never_won):,}')
print(f'\nTop 10 hottest numbers (recent 100 draws):')
print(num_df.nlargest(10, 'recent_100_wins')[['number','recent_100_wins','all_time_wins']].to_string(index=False))
print(f'\nTop 10 most overdue numbers:')
print(num_df.nlargest(10, 'draws_since_seen')[['number','draws_since_seen','all_time_wins']].to_string(index=False))

---
## 🔮 BLOCK 5 — Lucky Event Number Processing
Takes the numbers you entered in Block 1 and generates related picks:  
- **Exact number** itself  
- **Permutations** of its digits (all orderings)  
- **Digit-echo** numbers (using dominant digits from your events)  
- **Mirror** and **reverse** variants

In [ ]:
def derive_lucky_numbers(events: list) -> list:
    """
    From a list of (number_str, description) tuples,
    derive a pool of related 4D numbers to bet on.
    
    Derivation methods:
      1. Exact number as-is
      2. All digit permutations (up to 24 for 4 digits)
      3. Reverse of the number
      4. Digit shift: rotate digits left/right
      5. Mirror pairs: first two digits mirrored
      6. Sum echo: sum all digits, repeat to form 4-digit
    """
    derived = []

    for raw_num, desc in events:
        n = str(raw_num).zfill(4)[:4]   # ensure 4-digit string
        digits = list(n)

        # 1. Exact
        derived.append((n, f'Exact — {desc}'))

        # 2. Reverse
        rev = n[::-1]
        if rev != n:
            derived.append((rev, f'Reverse of {n} — {desc}'))

        # 3. All permutations (unique only)
        perms = set(''.join(p) for p in permutations(digits))
        perms.discard(n)
        perms.discard(rev)
        for p in sorted(perms)[:6]:   # cap at 6 permutations
            derived.append((p, f'Permutation of {n} — {desc}'))

        # 4. Rotate left: ABCD → BCDA
        rot_l = digits[1:] + [digits[0]]
        derived.append((''.join(rot_l), f'Rotate-left {n} — {desc}'))

        # 5. Rotate right: ABCD → DABC
        rot_r = [digits[-1]] + digits[:-1]
        derived.append((''.join(rot_r), f'Rotate-right {n} — {desc}'))

        # 6. Mirror: ABCD → ABBA
        mirror = digits[:2] + digits[:2][::-1]
        derived.append((''.join(mirror), f'Mirror {n} — {desc}'))

        # 7. Digit sum echo: sum digits, pad/repeat to 4
        dsum = sum(int(d) for d in digits)
        echo = str(dsum % 10) * 4
        derived.append((echo, f'Digit-sum echo of {n} ({dsum}→{dsum%10}) — {desc}'))

    # Deduplicate preserving first occurrence
    seen = set()
    unique = []
    for num, reason in derived:
        if num not in seen and len(num) == 4 and num.isdigit():
            seen.add(num)
            unique.append((num, reason))

    return unique


lucky_derived = derive_lucky_numbers(LUCKY_EVENTS)

if lucky_derived:
    print(f'✅ {len(lucky_derived)} lucky-derived numbers generated:')
    print(f'   {"Number":<8} {"Reason"}')
    print('   ' + '─' * 60)
    for num, reason in lucky_derived[:30]:   # show first 30
        hist = win_counter.get(num, 0)
        print(f'   {num:<8} {reason:<50} (historical wins: {hist})')
    if len(lucky_derived) > 30:
        print(f'   ... and {len(lucky_derived)-30} more')
else:
    print('ℹ️  No lucky events entered in Block 1.')
    print('   Lucky-derived picks will be skipped in final output.')
    print('   Add numbers to LUCKY_EVENTS in Block 1 to activate this feature.')

---
## 🎲 BLOCK 6 — Number Generation (All Methods)
Four independent pick pools, each using a different strategy:  

| Pool | Method | Logic |
|---|---|---|
| **Pure Random** | `np.random.randint` | No bias, completely uniform |
| **Stat Weighted** | Per-position digit weights | Slightly favours historically hotter digits |
| **Hot Numbers** | Recent 100-draw frequency | Numbers that have been appearing lately |
| **Overdue** | Draws since last seen | Numbers on a long cold streak |
| **Lucky Derived** | From your LUCKY_EVENTS | Personal meaningful numbers + variants |

In [ ]:
def gen_pure_random(n: int) -> list:
    """Completely uniform random — every number 0000-9999 equally likely."""
    picks = set()
    while len(picks) < n:
        picks.add(f'{np.random.randint(0, 10000):04d}')
    return sorted(picks)


def gen_stat_weighted(n: int, weights: dict) -> list:
    """
    Generate numbers where each digit position uses its historical
    frequency distribution as sampling weight.
    Slightly biases towards digits that have appeared more often.
    """
    picks = set()
    while len(picks) < n:
        digits = ''.join(
            str(np.random.choice(10, p=weights[pos]))
            for pos in range(4)
        )
        picks.add(digits)
    return sorted(picks)


def gen_from_pool(pool: list, n: int) -> list:
    """Sample n unique numbers from a given pool."""
    if not pool:
        return []
    size = min(n, len(pool))
    idx  = np.random.choice(len(pool), size, replace=False)
    return [pool[i] for i in sorted(idx)]


def gen_from_lucky(derived: list, n: int) -> list:
    """Pick top n from lucky-derived pool, prioritising exact event numbers."""
    if not derived:
        return []
    # Exact matches first, then permutations, then others
    exact = [item for item in derived if 'Exact' in item[1]]
    rest  = [item for item in derived if 'Exact' not in item[1]]
    ordered = exact + rest
    return [(num, reason) for num, reason in ordered[:n]]


# ── Generate all pools ───────────────────────────────────────────────────────
pool_pure_random   = gen_pure_random(N_PURE_RANDOM)
pool_stat_weighted = gen_stat_weighted(N_STAT_WEIGHTED, pos_weights)
pool_hot           = gen_from_pool(hot_numbers,     N_HOT_NUMBERS)
pool_overdue       = gen_from_pool(overdue_numbers, N_OVERDUE)
pool_lucky         = gen_from_lucky(lucky_derived,  N_LUCKY_DERIVED)

print('✅ All pools generated:')
print(f'   Pure random    : {pool_pure_random}')
print(f'   Stat weighted  : {pool_stat_weighted}')
print(f'   Hot numbers    : {pool_hot}')
print(f'   Overdue        : {pool_overdue}')
print(f'   Lucky derived  : {[x[0] for x in pool_lucky]}')

---
## 🏆 BLOCK 7 — Weekly Output: All Your 4D Picks

In [ ]:
def print_pool(title: str, emoji: str, pool, win_counter: Counter,
               recent_counter: Counter, lucky_mode=False):
    print(f'\n  {emoji} {title}')
    print('  ' + '─' * 65)
    items = pool if lucky_mode else [(n, '') for n in pool]
    for num, reason in items:
        hist   = win_counter.get(num, 0)
        recent = recent_counter.get(num, 0)
        tag    = ''
        if hist == 0:
            tag = '  [never won before]'
        elif recent > 0:
            tag = f'  [won {recent}x in last 100 draws]'
        note = f'  ← {reason}' if reason else ''
        print(f'  {num}    all-time: {hist:>4}  recent: {recent:>3}{tag}{note}')


print('=' * 70)
print(f'  🎱 4D WEEKLY PICKS')
print(f'  Generated : {datetime.now().strftime("%A, %d %B %Y %H:%M")}')
print(f'  Data      : {len(df):,} draws | '
      f'{df["draw_date"].min().date()} → {df["draw_date"].max().date()}')
print(f'  Bet days  : Wednesday · Saturday · Sunday')
print('=' * 70)

print_pool('PURE RANDOM',
           '🎲', pool_pure_random, win_counter, recent_counter)

print_pool('STATISTICALLY WEIGHTED (digit frequency)',
           '📊', pool_stat_weighted, win_counter, recent_counter)

print_pool('HOT NUMBERS (recent 100 draws)',
           '🔥', pool_hot, win_counter, recent_counter)

print_pool('OVERDUE NUMBERS (longest cold streak)',
           '❄️ ', pool_overdue, win_counter, recent_counter)

if pool_lucky:
    print_pool('LUCKY EVENT NUMBERS (from your real-life signs)',
               '🔮', pool_lucky, win_counter, recent_counter, lucky_mode=True)
else:
    print('\n  🔮 LUCKY EVENT NUMBERS')
    print('  ─' * 33)
    print('  (No lucky events entered — add to LUCKY_EVENTS in Block 1)')

print()
print('─' * 70)
print('  Prize structure (Ordinary $1 bet):')
print('  1st = $2,000  |  2nd = $1,000  |  3rd = $490')
print('  Starter = $250  |  Consolation = $60')
print('─' * 70)
print('  ⚠️  All numbers have equal 1-in-10,000 probability.')
print('      Hot/overdue/lucky picks do NOT have better odds.')
print('      They reflect pattern preference and personal meaning only.')
print('─' * 70)

# ── Save CSV ─────────────────────────────────────────────────────────────────
output_rows = []
for num in pool_pure_random:
    output_rows.append({'number': num, 'method': 'pure_random', 'reason': ''})
for num in pool_stat_weighted:
    output_rows.append({'number': num, 'method': 'stat_weighted', 'reason': ''})
for num in pool_hot:
    output_rows.append({'number': num, 'method': 'hot', 'reason': ''})
for num in pool_overdue:
    output_rows.append({'number': num, 'method': 'overdue', 'reason': ''})
for num, reason in pool_lucky:
    output_rows.append({'number': num, 'method': 'lucky', 'reason': reason})

out_df   = pd.DataFrame(output_rows)
os.makedirs('output', exist_ok=True)
out_file = f'output/4d_picks_{datetime.now().strftime("%Y%m%d_%H%M")}.csv'
out_df.to_csv(out_file, index=False)
print(f'\n💾 Saved: {out_file}')

---
## 📊 BLOCK 8 — Visual: Hot Zone Map of Recent 100 Draws
Which regions of the 4D number space (0000–9999) have been appearing recently?

In [ ]:
# Build 100×100 grid of recent win counts
grid = np.zeros((100, 100))
for num, count in recent_counter.items():
    if len(num) == 4 and num.isdigit():
        row = int(num[:2])
        col = int(num[2:])
        grid[row, col] = count

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# Left: recent hot zone
im1 = axes[0].imshow(grid, cmap='hot', aspect='auto')
plt.colorbar(im1, ax=axes[0], label='Times appeared (last 100 draws)')
axes[0].set_xlabel('Last 2 digits (00–99)')
axes[0].set_ylabel('First 2 digits (00–99)')
axes[0].set_title('🔥 Hot Zone Map\nLast 100 Draws', fontweight='bold')

# Mark your lucky event numbers on the map
for raw_num, desc in LUCKY_EVENTS:
    n = str(raw_num).zfill(4)[:4]
    if n.isdigit():
        r, c = int(n[:2]), int(n[2:])
        axes[0].plot(c, r, 'c*', markersize=15, label=f'{n}')
if LUCKY_EVENTS:
    axes[0].legend(title='Your lucky numbers', fontsize=8)

# Right: all-time frequency
grid_all = np.zeros((100, 100))
for num, count in win_counter.items():
    if len(num) == 4 and num.isdigit():
        row = int(num[:2])
        col = int(num[2:])
        grid_all[row, col] = count

im2 = axes[1].imshow(grid_all, cmap='Blues', aspect='auto')
plt.colorbar(im2, ax=axes[1], label='Times appeared (all time)')
axes[1].set_xlabel('Last 2 digits (00–99)')
axes[1].set_ylabel('First 2 digits (00–99)')
axes[1].set_title('📊 All-Time Frequency Map\nAll 4,479 Draws', fontweight='bold')

plt.suptitle('4D Number Space Frequency Maps', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('4d_v2_block8_hotmap.png', dpi=150, bbox_inches='tight')
plt.show()
print('📊 Maps saved as 4d_v2_block8_hotmap.png')

---
## 🔁 BLOCK 9 — Weekly Rerun Checklist
```
Before every 4D draw (Wed / Sat / Sun):

[ ] 1. Run scrape_4d.py to update data/4d_results.db with latest draw
[ ] 2. Update LUCKY_EVENTS in Block 1 with any new signs you've seen this week
       Examples of lucky events to record:
         - Car plate of a vehicle in an accident you witnessed
         - Temple fortune stick number (4-digit if available)
         - A number that appeared in a dream
         - Receipt/bill total that felt significant
         - House number of someone you visited
         - Bus/MRT number that kept appearing in your day
[ ] 3. Kernel → Restart & Run All
[ ] 4. Pick from whichever pool resonates with you this week
[ ] 5. CSV auto-saved with timestamp

Tip: You don't have to bet all pools every week.
     If you have strong lucky events → weight your bets toward Lucky Derived.
     If no events → Pure Random or Stat Weighted.
     Hot/Overdue are curiosity plays — streaks and droughts are random noise.
```

In [ ]:
conn.close()
print('✅ All done. DB connection closed.')
print(f'Next 4D draw days: Wednesday · Saturday · Sunday')
print('Rerun anytime: Kernel → Restart & Run All')